In [1]:
%%capture
!pip install duckdb jupyter

In [2]:
import duckdb

In [3]:
conn = duckdb.connect("../lipid_myth_buster.duckdb")

In [8]:
conn.sql("select count(*) as total_rows from ingestion.cac_health_screening")

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│       1688 │
└────────────┘

In [9]:
conn.sql("DESCRIBE ingestion.cac_health_screening")

┌───────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │       column_type        │  null   │   key   │ default │  extra  │
│      varchar      │         varchar          │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ no_figshare       │ BIGINT                   │ NO      │ NULL    │ NULL    │ NULL    │
│ test1_replcation2 │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ sbp               │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ dbp               │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ weight            │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ bmi               │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ glucose           │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ chol              │

In [12]:
conn.sql("select * from ingestion.cac_health_screening order by cacs desc limit 10")

┌─────────────┬───────────────────┬────────┬────────┬────────┬────────┬─────────┬────────┬────────────┬───────────────┬────────┬──────────┬──────────┬────────┬────────┬──────────────┬──────────┬───────────┬────────────────┬────────┬─────────┬────────┬───────────────────────────────┐
│ no_figshare │ test1_replcation2 │  sbp   │  dbp   │ weight │  bmi   │ glucose │  chol  │ creatinine │ idms_mdrd_gfr │   tg   │ hdl_chol │ ldl_chol │ hs_crp │ hb_a1c │ hypertension │ diabetes │ ex_smoker │ current_smoker │ age_yr │   sex   │  cacs  │         extracted_at          │
│    int64    │       int64       │ double │ double │ double │ double │ double  │ double │   double   │    double     │ double │  double  │  double  │ double │ double │    int64     │  int64   │   int64   │     int64      │ int64  │ varchar │ double │   timestamp with time zone    │
├─────────────┼───────────────────┼────────┼────────┼────────┼────────┼─────────┼────────┼────────────┼───────────────┼────────┼──────────┼─────────

In [4]:
conn.sql("SELECT * FROM staging.stg_cac_health_screening LIMIT 10")

┌─────────────┬────────┬─────────┬────────┬────────┬────────┬────────┬─────────┬────────┬────────────┬───────────────┬────────┬──────────┬──────────┬────────┬────────┬──────────────┬──────────┬────────┬───────────────────┬─────────────────┬────────────────────┬──────────────┬─────────┬───────┬─────────────────────────┬────────────────┬──────────────┬────────────────────────────────┬──────────────┬──────────────────────────┬─────────────────────────┬───────────────────────────────┐
│ no_figshare │ age_yr │   sex   │  sbp   │  dbp   │ weight │  bmi   │ glucose │  chol  │ creatinine │ idms_mdrd_gfr │   tg   │ hdl_chol │ ldl_chol │ hs_crp │ hb_a1c │ hypertension │ diabetes │  cacs  │ cac_risk_category │  ldl_category   │ high_triglycerides │ high_glucose │ low_hdl │ obese │ prediabetic_or_diabetic │ smoking_status │ tg_hdl_ratio │ estimated_ldl_particle_pattern │ non_hdl_chol │ metabolic_syndrome_score │ metabolic_health_status │         extracted_at          │
│    int64    │ int64  │ var

In [5]:
conn.sql("SELECT cac_risk_category, count(*) FROM staging.stg_cac_health_screening GROUP BY 1")

┌───────────────────┬──────────────┐
│ cac_risk_category │ count_star() │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Moderate          │          146 │
│ Mild              │          127 │
│ No calcification  │         1321 │
│ Severe            │           90 │
│ Unknown           │            4 │
└───────────────────┴──────────────┘

In [4]:
conn.sql("SELECT * FROM reports.cac_by_risk_group ORDER BY ldl_category")

┌─────────────────┬─────────────────────────┬───────────────┬──────────┬─────────────┬───────────────────────┬────────────────────────┬───────────────────┬─────────────┬─────────┬───────────┬──────────────────┐
│  ldl_category   │ metabolic_health_status │ subject_count │ avg_cacs │ median_cacs │ pct_any_calcification │ pct_moderate_or_severe │ avg_triglycerides │ avg_glucose │ avg_hdl │ avg_hba1c │ avg_tg_hdl_ratio │
│     varchar     │         varchar         │     int64     │  double  │   double    │        double         │         double         │      double       │   double    │ double  │  double   │      double      │
├─────────────────┼─────────────────────────┼───────────────┼──────────┼─────────────┼───────────────────────┼────────────────────────┼───────────────────┼─────────────┼─────────┼───────────┼──────────────────┤
│ Borderline high │ Metabolically healthy   │           130 │    39.63 │         0.0 │                  16.9 │                    8.5 │              89.6 │ 

In [5]:
conn.sql("SELECT * FROM reports.cac_by_age ORDER BY age_group")

┌───────────┬─────────────────────────┬───────────────┬──────────┬─────────────┬───────────────────────┬────────────────────────┬─────────┬───────────────────┬──────────────────┬───────────────┐
│ age_group │ metabolic_health_status │ subject_count │ avg_cacs │ median_cacs │ pct_any_calcification │ pct_moderate_or_severe │ avg_ldl │ avg_triglycerides │ avg_tg_hdl_ratio │ avg_met_score │
│  varchar  │         varchar         │     int64     │  double  │   double    │        double         │         double         │ double  │      double       │      double      │    double     │
├───────────┼─────────────────────────┼───────────────┼──────────┼─────────────┼───────────────────────┼────────────────────────┼─────────┼───────────────────┼──────────────────┼───────────────┤
│ 30-39     │ Metabolically healthy   │            23 │    16.11 │         0.0 │                  13.0 │                    4.3 │   105.8 │              75.0 │             1.36 │           0.0 │
│ 30-39     │ Metabolical

In [6]:
conn.sql("SELECT * FROM reports.ldl_particle_analysis")

┌────────────────────────────────┬─────────────────┬───────────────┬──────────┬─────────────┬───────────────────────┬────────────────────────┬──────────────────┬──────────────────┐
│ estimated_ldl_particle_pattern │  ldl_category   │ subject_count │ avg_cacs │ median_cacs │ pct_any_calcification │ pct_moderate_or_severe │ avg_tg_hdl_ratio │ avg_non_hdl_chol │
│            varchar             │     varchar     │     int64     │  double  │   double    │        double         │         double         │      double      │      double      │
├────────────────────────────────┼─────────────────┼───────────────┼──────────┼─────────────┼───────────────────────┼────────────────────────┼──────────────────┼──────────────────┤
│ Pattern A (Large Buoyant)      │ Borderline high │           232 │    42.94 │         0.0 │                  17.2 │                   10.3 │             1.67 │            157.9 │
│ Pattern A (Large Buoyant)      │ High            │            76 │    27.82 │         0.0 │  